In [2]:
from pathlib import Path

import pandas as pd

traffic_path = Path("../data/raw/tmc_summary_data.csv")

traffic_df = pd.read_csv(traffic_path, low_memory=False)

print("Dataset shape:", traffic_df.shape)
print("\nColumns and data types:")
print(traffic_df.dtypes.to_string())

display(traffic_df.head(3))

Dataset shape: (30953, 34)

Columns and data types:
_id                    int64
count_id               int64
count_date               str
location_name            str
longitude            float64
latitude             float64
centreline_type        int64
centreline_id          int64
px                   float64
count_duration           str
total_vehicle          int64
total_bike             int64
total_heavy_pct      float64
total_pedestrian       int64
am_peak_start            str
am_peak_vehicle        int64
am_peak_bike           int64
am_peak_heavy_pct    float64
pm_peak_start            str
pm_peak_vehicle        int64
pm_peak_bike           int64
pm_peak_heavy_pct    float64
n_appr_vehicle         int64
n_appr_bike            int64
n_appr_heavy_pct     float64
e_appr_vehicle         int64
e_appr_bike            int64
e_appr_heavy_pct     float64
s_appr_vehicle         int64
s_appr_bike            int64
s_appr_heavy_pct     float64
w_appr_vehicle         int64
w_appr_bike         

,_id,count_id,count_date,location_name,longitude,latitude,centreline_type,centreline_id,px,count_duration,...,n_appr_heavy_pct,e_appr_vehicle,e_appr_bike,e_appr_heavy_pct,s_appr_vehicle,s_appr_bike,s_appr_heavy_pct,w_appr_vehicle,w_appr_bike,w_appr_heavy_pct
0,1,117135,2026-09-29,Marlee Ave / Glengrove Ave,-79.444490,43.710732,2,13456552,NaN,14,...,0.033,672,11,0.018,7136,88,0.029,1024,24,0.012
1,2,117708,2026-09-26,Maitland St / Church St,-79.380491,43.664641,2,13464377,2016.0,14,...,0.024,2482,144,0.010,4205,433,0.020,19,58,0.000
2,3,117710,2026-09-26,Wellesley St E / Church St,-79.380932,43.665706,2,13464212,24.0,14,...,0.024,4346,605,0.030,4038,442,0.019,2842,490,0.051


In [3]:

traffic_df["count_date"] = pd.to_datetime(
    traffic_df["count_date"],
    errors="coerce"
)

years = traffic_df["count_date"].dt.year

print("Earliest date:", traffic_df["count_date"].min())
print("Latest date:", traffic_df["count_date"].max())

print("\nMeasurements by period:")
print("Before 2018:", (years < 2018).sum())
print("During 2018:", (years == 2018).sum())
print("After 2018:", (years > 2018).sum())
print("Missing dates:", years.isna().sum())


Earliest date: 1984-01-03 00:00:00
Latest date: 2026-09-29 00:00:00

Measurements by period:
Before 2018: 20732
During 2018: 823
After 2018: 9398
Missing dates: 0


In [4]:

signals_df = pd.read_csv(
    "../data/raw/traffic-signals.csv"
)

traffic_history = traffic_df.loc[
    traffic_df["count_date"] < "2019-01-01"
].copy()

print("Traffic volume locations:")
display(
    traffic_history[
        ["location_name", "centreline_id", "count_date", "total_vehicle"]
    ].head(8)
)

print("Traffic Signals intersections:")
display(
    signals_df[
        ["id", "mainStreet", "side1Street", "side2Street"]
    ].head(8)
)


Traffic volume locations:


,location_name,centreline_id,count_date,total_vehicle
128,Milverton Blvd / Marlow Ave,13460917,2006-02-28,504
6174,Royal Dr,60023807,2010-06-02,18300
9400,Queen St E / Jones Ave / Leslie Grove Park Trl,13464597,2018-12-20,8539
9401,Bloor St W: Grenview Blvd S - Jackson Ave,8292238,2018-12-20,14303
9402,Queen St E / Eastern Ave,13463876,2018-12-20,6304
9403,Eglinton Ave W / Hwy 27 / Hwy 401 / Hwy 427 Eg...,13463207,2018-12-20,33754
9404,Queen St E / Wineva Ave (West),13463118,2018-12-20,9623
9405,Cedar Brae Blvd / Banmoor Blvd / Bellamy Rd N,13449721,2018-12-20,7216


Traffic Signals intersections:


,id,mainStreet,side1Street,side2Street
0,2,JARVIS ST,FRONT ST E,NaN
1,3,KING ST E,JARVIS ST,NaN
2,4,JARVIS ST,ADELAIDE ST E,NaN
3,5,JARVIS ST,RICHMOND ST E,NaN
4,6,JARVIS ST,QUEEN ST E,NaN
5,7,JARVIS ST,SHUTER ST,NaN
6,8,JARVIS ST,DUNDAS ST E,NaN
7,9,JARVIS ST,GERRARD ST E,NaN


In [5]:

import re


def normalize_street(street):
    if pd.isna(street):
        return None

    street = str(street).upper().strip()
    street = re.sub(r"\s+", " ", street)
    street = re.sub(r"\bAV\b", "AVE", street)
    street = re.sub(r"\bBV\b", "BLVD", street)

    return street


def make_intersection_key(street_a, street_b):
    a = normalize_street(street_a)
    b = normalize_street(street_b)

    if a is None or b is None:
        return None

    return " | ".join(sorted([a, b]))


# 1. Ключи для Traffic Signals
signals_df["intersection_key"] = signals_df.apply(
    lambda row: make_intersection_key(
        row["mainStreet"],
        row["side1Street"]
    ),
    axis=1
)

# 2. Оставляем измерения с двумя улицами
traffic_two = traffic_history.loc[
    traffic_history["location_name"].fillna("").str.count("/") == 1
].copy()

streets = traffic_two["location_name"].str.split(
    "/", n=1, expand=True
)

traffic_two["intersection_key"] = [
    make_intersection_key(a, b)
    for a, b in zip(streets[0], streets[1])
]

# 3. Исключаем неоднозначные ключи светофоров
key_counts = signals_df["intersection_key"].value_counts()

unique_signals = signals_df.loc[
    signals_df["intersection_key"].map(key_counts).eq(1),
    ["intersection_key", "id"]
]

# 4. Сопоставляем измерения с ID светофоров
traffic_matched = traffic_two.merge(
    unique_signals,
    on="intersection_key",
    how="inner",
    validate="many_to_one"
)

print("Two-street measurements:", len(traffic_two))
print("Matched measurements:", len(traffic_matched))
print("Matched intersections:", traffic_matched["id"].nunique())

display(
    traffic_matched[
        ["location_name", "id", "count_date", "total_vehicle"]
    ].head(5)
)


Two-street measurements: 16052
Matched measurements: 10526
Matched intersections: 1512


,location_name,id,count_date,total_vehicle
0,Bloor St W / Royal York Rd,338,2018-12-20,24550
1,Queen St E / Greenwood Ave,1393,2018-12-20,8144
2,Bloor St W / Aukland Rd,1516,2018-12-20,14222
3,Queen St E / Leslie St,547,2018-12-20,10247
4,Bloor St W / Montgomery Rd,1591,2018-12-20,16924


In [6]:

# Measurements available before 2018
traffic_before_2018 = traffic_matched.loc[
    traffic_matched["count_date"] < "2018-01-01"
].copy()

# Intersections from our W04 dataset
w4_ids = pd.read_csv(
    "../data/processed/intersection_day_2018.csv",
    usecols=["intersection_id"]
)["intersection_id"].unique()

covered_ids = set(w4_ids) & set(traffic_before_2018["id"])

print("W04 intersections:", len(w4_ids))
print("Covered before 2018:", len(covered_ids))
print("Coverage:", f"{len(covered_ids) / len(w4_ids):.1%}")

print("\nCount duration distribution:")
print(
    traffic_before_2018["count_duration"]
    .value_counts(dropna=False)
    .head(10)
)


W04 intersections: 2258
Covered before 2018: 1369
Coverage: 60.6%

Count duration distribution:
count_duration
8R    9795
8S     411
Name: count, dtype: int64


In [7]:

# 1. Load the W04 modeling dataset
model_df = pd.read_csv(
    "../data/processed/intersection_day_2018.csv",
    parse_dates=["prediction_date", "target_date"]
)

# 2. Select comparable historical measurements
traffic_reference = (
    traffic_before_2018.loc[
        traffic_before_2018["count_duration"].eq("8R")
        & traffic_before_2018["total_vehicle"].notna(),
        ["id", "count_date", "total_vehicle"]
    ]
    .groupby(["id", "count_date"], as_index=False)["total_vehicle"]
    .mean()
    .sort_values(["count_date", "id"])
    .drop_duplicates(subset="id", keep="last")
    .rename(columns={
        "id": "intersection_id",
        "count_date": "traffic_measurement_date",
        "total_vehicle": "historical_traffic_volume"
    })
)

# 3. Join traffic measurements to intersection-day rows
model_df = model_df.merge(
    traffic_reference,
    on="intersection_id",
    how="left",
    validate="many_to_one"
)

# 4. Add calendar feature
model_df["month"] = model_df["target_date"].dt.month

# 5. Show the modeling features
feature_cols = [
    "historical_ksi_count",
    "historical_traffic_volume",
    "day_of_week",
    "month"
]

print("Dataset shape:", model_df.shape)
print("Available traffic measurements:",
      model_df["historical_traffic_volume"].notna().sum())

display(model_df[feature_cols + ["target"]].head())


Dataset shape: (823217, 9)
Available traffic measurements: 498011


,historical_ksi_count,historical_traffic_volume,day_of_week,month,target
0,0,12960.0,1,1,0
1,0,12960.0,2,1,0
2,0,12960.0,3,1,0
3,0,12960.0,4,1,0
4,0,12960.0,5,1,0


In [8]:
export_columns = [
    "intersection_id",
    "prediction_date",
    "target_date",
    "historical_ksi_count",
    "historical_traffic_volume",
    "day_of_week",
    "month",
    "target"
]

output_path = Path(
    "../data/processed/intersection_day_2018_w5_features.csv"
)

model_df[export_columns].to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Shape:", model_df[export_columns].shape)

Saved: ..\data\processed\intersection_day_2018_w5_features.csv
Shape: (823217, 8)


In [9]:

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Define features and target
feature_cols = [
    "historical_ksi_count",
    "historical_traffic_volume",
    "day_of_week",
    "month"
]

X = model_df[feature_cols].copy()
y = model_df["target"].copy()

# 2. Select numerical features
numeric_features = [
    "historical_ksi_count",
    "historical_traffic_volume"
]

# 3. Build numerical preprocessing pipeline
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# 4. Select categorial features
categorical_features = [
    "day_of_week", 
    "month"
]

# 5. Build categorical preprocessing pipeline
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


print(numeric_pipeline)
print(categorical_pipeline)



Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])
Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore'))])


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

model = make_pipeline(preprocessor, LogisticRegression(max_iter=1000))

In [11]:

# Temporal train/validation split
train_mask = model_df["prediction_date"] <= "2018-09-29"
valid_mask = model_df["prediction_date"] >= "2018-10-01"

X_train = X.loc[train_mask]
X_valid = X.loc[valid_mask]

y_train = y.loc[train_mask]
y_valid = y.loc[valid_mask]

print("Train observations:", len(X_train))
print("Validation observations:", len(X_valid))

print("Train positives:", y_train.sum())
print("Validation positives:", y_valid.sum())


Train observations: 613223
Validation observations: 207736
Train positives: 81
Validation positives: 38


In [12]:
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('columntransformer', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](4,)","['historical_ksi_count','historical_traffic_volume','day_of_week','month']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,4
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``

In [13]:
from sklearn.metrics import average_precision_score

y_proba = model.predict_proba(X_valid)[:, 1]

average_precision_score(y_valid, y_proba)

0.0003719788062030399

In [14]:

from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(
    n_splits=3,
    gap=1
)


In [15]:

import numpy as np

# 1. Get training dates
train_dates = model_df.loc[X_train.index, "prediction_date"]
unique_dates = pd.Index(train_dates.unique()).sort_values()

# 2. Convert date splits into row indices
cv_splits = []

for train_days_idx, valid_days_idx in tscv.split(unique_dates):
    train_days = unique_dates[train_days_idx]
    valid_days = unique_dates[valid_days_idx]

    train_idx = np.flatnonzero(
        train_dates.isin(train_days).to_numpy()
    )

    valid_idx = np.flatnonzero(
        train_dates.isin(valid_days).to_numpy()
    )

    cv_splits.append((train_idx, valid_idx))

# 3. Essential check: positive samples in each fold
for i, (train_idx, valid_idx) in enumerate(cv_splits, start=1):
    print(
        f"Fold {i}: "
        f"train positives={y_train.iloc[train_idx].sum()}, "
        f"validation positives={y_train.iloc[valid_idx].sum()}"
    )


Fold 1: train positives=14, validation positives=20
Fold 2: train positives=34, validation positives=18
Fold 3: train positives=51, validation positives=29


In [16]:
from sklearn.model_selection import cross_validate

cv_results = cross_validate(model, X_train, y_train, cv=cv_splits, scoring="average_precision")

print(cv_results["test_score"])

[0.00049171 0.00041989 0.00096609]


## Week 05 — Results and Conclusions

### Objective

Extend the Week 04 Logistic Regression baseline by introducing numerical and categorical preprocessing within a unified scikit-learn Pipeline.

The prediction task remains binary classification of next-day matched KSI collision events at signalized intersections in Toronto.

### Feature Engineering

The model uses four features:

| Feature | Type |
|---|---|
| `historical_ksi_count` | Numerical |
| `historical_traffic_volume` | Numerical |
| `day_of_week` | Categorical |
| `month` | Categorical |

Historical traffic measurements were matched to Traffic Signals intersection IDs using normalized street names. Only selected measurements dated before 2018 were used.

Traffic volume is available for approximately 60.5% of modeling observations.

### ML Pipeline

Numerical preprocessing:
- `SimpleImputer(strategy="median")`
- `StandardScaler`

Categorical preprocessing:
- `SimpleImputer(strategy="most_frequent")`
- `OneHotEncoder(handle_unknown="ignore")`

Both branches are combined using `ColumnTransformer`, followed by `LogisticRegression`.

The fitted Pipeline produces 18 transformed features.

### Validation Strategy

A chronological train/validation split was used:

- **Training:** January 1 – September 29, 2018
- **Gap:** September 30, 2018
- **Validation:** October 1 – December 31, 2018

Three additional temporal cross-validation folds were constructed from the training period using `TimeSeriesSplit` on unique prediction dates with a one-day gap.

Preprocessing is fitted separately within each training fold to prevent validation-data leakage.

### Experimental Results

| Evaluation | Average Precision |
|---|---:|
| Temporal Fold 1 | 0.000492 |
| Temporal Fold 2 | 0.000420 |
| Temporal Fold 3 | 0.000966 |
| **Mean Temporal CV** | **0.000626** |
| Separate Validation | 0.000372 |

### Interpretation

The Pipeline successfully combines numerical and categorical features and performs temporal cross-validation.

Average Precision varies considerably between folds, indicating that the observed ranking performance is unstable across time periods.

The results are exploratory and do not establish an improvement over the Week 04 baseline.

### Limitations

- KSI collisions are extremely rare, resulting in very few positive observations.
- Traffic volume measurements are incomplete and may be outdated.
- The exact meaning of the selected `8R` count-duration code and historical data availability require further verification.
- Intersection matching based on street names is incomplete and may contain errors.
- The model is evaluated only on 2018 development data; no untouched final test evaluation has been performed.
- Predicted probabilities have not been calibrated.

### Conclusion

Week 05 establishes a working numerical and categorical preprocessing Pipeline with temporal cross-validation.

Further work should focus on validating historical feature availability, improving data coverage, and comparing models using consistent evaluation periods and metrics.